In [ ]:
import subprocess
import time
import matplotlib.pyplot as plt
import csv
import os
import statistics

# --- CONFIGURAZIONE ---
# Assicurati che il tuo eseguibile CUDA accetti il block_size come ultimo argomento
EXECUTABLE = './../median_filter_cuda' 
INPUT_IMG  = '../noise_1920x1280.ppm'
OUTPUT_IMG = 'result_cuda.ppm'

# 8 Configurazioni: dai 32 (dimensione Warp) ai 1024 (limite Tesla T4)
BLOCK_SIZES = [32, 64, 128, 256, 448, 512, 768, 1024]
RUNS_PER_TEST = 30  
CSV_FILENAME = 'benchmark_cuda_results.csv'
# ---------------------

program_name = EXECUTABLE.split('/')[-1]

print("🚀 Riscaldamento della GPU (CUDA Context Creation)...")
# Il primo lancio in CUDA è sempre lento a causa della creazione del contesto
subprocess.run([EXECUTABLE, INPUT_IMG, OUTPUT_IMG, '256'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

mean_execution_times = []
write_header = not os.path.isfile(CSV_FILENAME)

with open(CSV_FILENAME, mode='a', newline='') as csv_file:
    writer = csv.writer(csv_file)
    if write_header:
        writer.writerow(['Program Name', 'Input Name', 'Threads Per Block', 'Repetition', 'Time (Seconds)'])

    print(f"\n🔥 Inizio Benchmark CUDA ({RUNS_PER_TEST} run per configurazione)...\n")

    for block_size in BLOCK_SIZES:
        print(f"Testando Block Size: {block_size}...", end="", flush=True)
        
        times_for_this_config = []
        
        for rep in range(RUNS_PER_TEST):
            # Passiamo il block_size come argomento all'eseguibile
            cmd = [EXECUTABLE, INPUT_IMG, OUTPUT_IMG, str(block_size)]
            
            start_time = time.perf_counter()
            subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            end_time = time.perf_counter()
            
            run_time = end_time - start_time
            times_for_this_config.append(run_time)
            
            writer.writerow([program_name, INPUT_IMG, block_size, rep + 1, run_time])
            csv_file.flush() 
        
        mean_time = statistics.mean(times_for_this_config)
        mean_execution_times.append(mean_time)
        print(f" Media: {mean_time:.5f}s")

# --- GENERAZIONE GRAFICO ---
plt.figure(figsize=(10, 6))
plt.plot(BLOCK_SIZES, mean_execution_times, marker='s', markersize=8, 
         linestyle='-', color='#28a745', linewidth=2, label='CUDA Mean Execution Time')

plt.title(f'CUDA Performance Scaling: {program_name}', fontsize=14, fontweight='bold')
plt.xlabel('Threads Per Block', fontsize=12)
plt.ylabel('Execution Time (Seconds)', fontsize=12)
plt.xticks(BLOCK_SIZES)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=11)

# Annotazione per evidenziare il punto migliore
min_time = min(mean_execution_times)
best_block = BLOCK_SIZES[mean_execution_times.index(min_time)]
plt.annotate(f'Best: {best_block}', xy=(best_block, min_time), xytext=(best_block, min_time + 0.01),
             arrowprops=dict(facecolor='black', shrink=0.05), ha='center')

plt.tight_layout()
graph_filename = f'cuda_scaling_{program_name}.png'
plt.savefig(graph_filename, dpi=300)
print(f"\n✅ Benchmark completato! Grafico salvato come '{graph_filename}'")